# Build NetCDF from CSV metadata and measurements

This notebook reads the metadata CSV, creates an xarray Dataset with dimensions `Time` and `RGIId`, populates variables from a long-form measurements table, and writes a compressed NetCDF file.

In [ ]:
import pandas as pd
import xarray as xr
import numpy as np

meta = pd.read_csv("out.csv", index_col=None)
meta = meta.loc[:, meta.columns != ""]

time = pd.to_datetime(["2000-01-01", "2000-02-01", "2000-03-01"])
rgiid = ["RGI60-01.00001", "RGI60-01.00002"]

ds = xr.Dataset(coords={"Time": ("Time", time), "RGIId": ("RGIId", rgiid)})
ds.Time.attrs["standard_name"] = "time"
ds.Time.attrs["calendar"] = "gregorian"
ds.RGIId.attrs["description"] = "Randolph Glacier Inventory ID"


In [ ]:
for _, row in meta.iterrows():
    var = str(row["var_short_name"]).strip()
    if not var or var.lower() in {"time", "rgiid"}:
        continue
    dtype_hint = str(row.get("type", "")).lower()
    if dtype_hint.startswith("long") or dtype_hint.startswith("int"):
        dtype = np.int64
        fill_value = -9999
        arr = np.full((len(time), len(rgiid)), fill_value, dtype=dtype)
        data = xr.DataArray(arr, dims=("Time", "RGIId"), coords={"Time": time, "RGIId": rgiid})
        data.attrs["_FillValue"] = fill_value
    else:
        dtype = np.float32
        arr = np.full((len(time), len(rgiid)), np.nan, dtype=dtype)
        data = xr.DataArray(arr, dims=("Time", "RGIId"), coords={"Time": time, "RGIId": rgiid})
    data.attrs["long_name"] = row.get("var_long_name", "")
    data.attrs["units"] = row.get("unit", "")
    data.attrs["comment"] = row.get("comment", "")
    ds[var] = data


In [ ]:
long_df = pd.DataFrame({
    "Time": pd.to_datetime(["2000-01-01","2000-01-01","2000-02-01"]),
    "RGIId": ["RGI60-01.00001","RGI60-01.00002","RGI60-01.00001"],
    "var_short_name": ["area_annual","area_annual","area_annual"],
    "value": [1.23, 2.34, 1.50]
})

for var in long_df["var_short_name"].unique():
    sub = long_df[long_df["var_short_name"] == var].copy()
    sub = sub.groupby(["Time", "RGIId"], as_index=False)["value"].mean()
    pivot = sub.pivot(index="Time", columns="RGIId", values="value")
    pivot = pivot.reindex(index=time, columns=rgiid)
    if var in ds:
        ds[var].values[:] = pivot.values
    else:
        ds[var] = xr.DataArray(pivot.values, dims=("Time", "RGIId"), coords={"Time": time, "RGIId": rgiid})


In [ ]:
encoding = {}
for var in ds.data_vars:
    if ds[var].dtype.kind == "f":
        encoding[var] = {"dtype": "float32", "zlib": True, "complevel": 4, "_FillValue": np.float32(np.nan)}
    elif ds[var].dtype.kind == "i":
        encoding[var] = {"dtype": "int32", "zlib": True, "complevel": 4, "_FillValue": -9999}
    else:
        encoding[var] = {}

ds.to_netcdf("glacier_data.nc", engine="netcdf4", encoding=encoding)

print("Wrote glacier_data.nc")
ds
